In [1]:
from torchvision import transforms
from transformers import FlavaModel,AutoProcessor,CLIPProcessor,CLIPModel
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from modules import FLAVAExtractor,HeadClassifierFLAVAModel,Train,CreationFLAVADataset,CreationProcessedDataset,FLAVACollateFunction,split_flava_embeddings,MemeDetector,Test

import sys, os
sys.path.append(os.path.abspath(".."))

from CLIP_model.modules import CreationProcessedDataset as CreationClipDataset
from common_files import creation_dataframe,set_seed,seed_worker


In [2]:
seed=42
set_seed(seed=seed)
generator=torch.Generator()
generator.manual_seed(seed)

In [3]:
#creation of the dataframes
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [4]:
#creation of the datasets used for the FLAVA forward
train_FLAVA_dataset=CreationFLAVADataset(train_df)
val_FLAVA_dataset=CreationFLAVADataset(val_df)

In [5]:
#processor used to process the input data before the FLAVA forward
processor=AutoProcessor.from_pretrained("facebook/flava-full")

In [6]:
flava_model=FlavaModel.from_pretrained("facebook/flava-full")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [7]:
collate_function_object=FLAVACollateFunction(processor)

In [8]:
#creation of the dataloaders for the FLAVA forward and process of the input data
train_FLAVA_dataloader=DataLoader(train_FLAVA_dataset,collate_fn=collate_function_object.collate_fn,batch_size=batch_size,shuffle=False,generator=generator,worker_init_fn=seed_worker)
val_FLAVA_dataloader=DataLoader(val_FLAVA_dataset,collate_fn=collate_function_object.collate_fn,batch_size=batch_size,shuffle=False,generator=generator,worker_init_fn=seed_worker)

In [9]:
flava_extractor=FLAVAExtractor(flava_model=flava_model,device=device)

In [9]:
#extraction of the FLAVA embeddings
#train_flava_data=flava_extractor.get_embeddings(train_FLAVA_dataloader,"./modules/flava_embeddings","train")
#val_flava_data=flava_extractor.get_embeddings(train_FLAVA_dataloader,"./modules/flava_embeddings","val")

In [10]:
train_flava_embeddings=torch.load("./modules/flava_embeddings/train_flava_embeddings.pt")
val_flava_embeddings=torch.load("./modules/flava_embeddings/val_flava_embeddings.pt")

In [11]:
train_clip_embeddings=torch.load("../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")
val_clip_embeddings=torch.load("../CLIP_model/modules/clip_embeddings/val_clip_embeddings.pt")

In [12]:
original_train_all_embeddings=train_flava_embeddings.copy()
original_train_all_embeddings.update(train_clip_embeddings)
val_all_embeddings=val_flava_embeddings.copy()
val_all_embeddings.update(val_clip_embeddings)

In [13]:
train_all_embeddings,test_all_embeddings=split_flava_embeddings(original_train_all_embeddings)

In [14]:
train_dataset=CreationProcessedDataset(train_all_embeddings)
val_dataset=CreationProcessedDataset(val_all_embeddings)
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,drop_last=True,generator=generator,worker_init_fn=seed_worker)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,drop_last=True,generator=generator,worker_init_fn=seed_worker)

In [15]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_all_embeddings["labels"].numpy())
class_weight=torch.tensor(class_weight, dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [16]:
#Training without CLIP embeddings and using FLAVA pooler embeddings
model=HeadClassifierFLAVAModel(fc_layer_sizes=[384],with_clip_image=False,with_clip_text=False)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=5e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/pooler_only_savings",multimodal=False,with_clip=False)

2026-04-05 22:16:34.165 | INFO     | modules.train:run_training:98 - Epoch 0 :
2026-04-05 22:16:44.410 | INFO     | modules.train:run_training:195 - Epoch 0: Train Loss = 0.714121593092276
2026-04-05 22:16:44.410 | INFO     | modules.train:run_training:196 - Epoch 0: Train Accuracy = 0.5330805439330544
2026-04-05 22:16:44.411 | INFO     | modules.train:run_training:197 - Epoch 0: Train F1 = 0.5341508347487461
2026-04-05 22:16:44.414 | INFO     | modules.train:run_training:199 - Epoch 0: Validation Loss = 0.7143698533376058
2026-04-05 22:16:44.415 | INFO     | modules.train:run_training:200 - Epoch 0: Validation Accuracy = 0.49583333333333335
2026-04-05 22:16:44.415 | INFO     | modules.train:run_training:201 - Epoch 0: Validation F1 = 0.32871402042711234
2026-04-05 22:16:44.664 | INFO     | modules.train:run_training:98 - Epoch 1 :
2026-04-05 22:16:47.743 | INFO     | modules.train:run_training:195 - Epoch 1: Train Loss = 0.7064857787168176
2026-04-05 22:16:47.744 | INFO     | modules.

In [ ]:
#Training with CLIP embeddings and using FLAVA pooler embeddings
model=HeadClassifierFLAVAModel(fc_layer_sizes=[1152],with_clip_image=True,with_clip_text=True)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/pooler_with_clip_savings",with_clip=True,multimodal=False)

2026-03-12 17:27:08.812 | INFO     | modules.train:run_training:114 - Epoch 0 :
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.7242236101402426
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.5679245283018868
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.547953923949787
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.7560108502705892
2026-03-12 17:27:12.256 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.49166666666666664
2026-03-12 17:27:12.256 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.4176503200276207
2026-03-12 17:27:12.351 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:27:15.677 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6407365142174487
2026-03-12 17:27:15.678 | INFO     | modules

In [20]:
#Training without CLIP embeddings and using FLAVA multimodal embeddings
model=HeadClassifierFLAVAModel()
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/multimodal_only_savings",multimodal=True)

2026-03-12 17:33:55.332 | INFO     | modules.train:run_training:114 - Epoch 0 :
2026-03-12 17:33:58.159 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.769842326866006
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.38254716981132075
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.28489077155826287
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.6614402770996094
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.5020833333333333
2026-03-12 17:33:58.164 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.34156378600823045
2026-03-12 17:33:58.197 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:34:01.352 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6918763295659479
2026-03-12 17:34:01.352 | INFO     | modul

In [21]:
#Training with CLIP embeddings and using FLAVA multimodal embeddings
model=HeadClassifierFLAVAModel(with_clip_image=True,with_clip_text=True)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/multimodal_with_clip_savings",multimodal=True,with_clip=True)

2026-03-12 17:34:26.966 | INFO     | modules.train:run_training:114 - Epoch 0 :


2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.7605600908117475
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.6127358490566037
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.5446854926126455
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.8256111224492391
2026-03-12 17:34:30.838 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.4979166666666667
2026-03-12 17:34:30.838 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.3431415414142217
2026-03-12 17:34:30.840 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:34:34.645 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6549331453611266
2026-03-12 17:34:34.645 | INFO     | modules.train:run_training:212 - Epoch 1: Train Accuracy = 0.61875
2026-03-12 17:34:34.

In [16]:
batch_size=32
test_dataset=CreationProcessedDataset(test_all_embeddings)
test_dataloader=DataLoader(test_dataset,batch_size=batch_size,shuffle=True,drop_last=True,generator=generator,worker_init_fn=seed_worker)

In [17]:
model=HeadClassifierFLAVAModel()
model_parameters=torch.load("./modules/train_savings/pooler_only_savings/model_state.pt")
model.load_state_dict(model_parameters)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
tester=Test(model,device,loss_fn)
f1,accuracy,final_loss,all_predictions,all_targets,logits=tester.run_testing(test_dataloader,with_scores=False)

RuntimeError: Error(s) in loading state_dict for HeadClassifierFLAVAModel:
	Missing key(s) in state_dict: "projection_clip_image.weight", "projection_clip_image.bias", "projection_clip_text.weight", "projection_clip_text.bias", "norm_proj_image.weight", "norm_proj_image.bias", "norm_proj_text.weight", "norm_proj_text.bias". 

In [ ]:
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")